# 01 Vehicle Detection

**Description:** Initialize a YOLOv8-based vehicle detection workflow on sample CCTV imagery or video frames.

**Objective:** Load media, run placeholder or real YOLO detection, and save tabular plus visual outputs into `outputs/vehicle_detection/`.

## Step 1. Environment Setup

This section imports core libraries and creates the output folder used by the notebook.

In [ ]:
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Outputs root: {OUTPUT_ROOT}')

from ultralytics import YOLO

OUTPUT_DIR = OUTPUT_ROOT / 'vehicle_detection'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = OUTPUT_DIR / 'detected_vehicles.csv'
ANNOTATED_FRAME_PATH = OUTPUT_DIR / 'frame_annotated.jpg'


## Step 2. Load Sample Frame

Replace the sample path with a real CCTV image or extract a frame from a video stream for live testing.

In [ ]:
sample_frame_path = PROJECT_ROOT / 'demo' / 'sample_cctv_frame.jpg'
frame = cv2.imread(str(sample_frame_path))

if frame is None:
    frame = np.full((480, 640, 3), 40, dtype=np.uint8)
    cv2.putText(frame, 'Sample CCTV Frame', (150, 240), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 255), 2)

frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 5))
plt.imshow(frame_rgb)
plt.title('Input CCTV Frame')
plt.axis('off')
plt.show()


## Step 3. Initialize YOLOv8

Swap the model path if you want to use the trained FlowX weights instead of the lightweight base model.

In [ ]:
model_path = PROJECT_ROOT / 'models' / 'traffic_detector.pt'
if not model_path.exists():
    model_path = PROJECT_ROOT / 'yolov8n.pt'

model = YOLO(str(model_path))
print(f'Loaded model: {model_path}')


## Step 4. Run Detection and Save Outputs

The example below supports a safe fallback if detections are unavailable. Replace the placeholder handling with your production thresholds or class filtering rules as needed.

In [ ]:
results = model.predict(source=frame, verbose=False)
records = []
annotated = frame.copy()

for result in results:
    if result.boxes is None:
        continue
    for box in result.boxes:
        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
        conf = float(box.conf[0].item())
        cls_id = int(box.cls[0].item())
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)
        records.append({
            'vehicle_id': len(records) + 1,
            'class_id': cls_id,
            'confidence': conf,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
            'centroid_x': cx,
            'centroid_y': cy,
        })
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.circle(annotated, (cx, cy), 4, (0, 0, 255), -1)

if not records:
    records = [
        {
            'vehicle_id': 1,
            'class_id': 2,
            'confidence': 0.95,
            'x1': 180,
            'y1': 220,
            'x2': 300,
            'y2': 320,
            'centroid_x': 240,
            'centroid_y': 270,
        }
    ]
    cv2.rectangle(annotated, (180, 220), (300, 320), (0, 255, 255), 2)

vehicle_df = pd.DataFrame(records)
vehicle_df.to_csv(CSV_PATH, index=False)
cv2.imwrite(str(ANNOTATED_FRAME_PATH), annotated)

print(f'Saved detections to {CSV_PATH}')
print(f'Saved annotated frame to {ANNOTATED_FRAME_PATH}')
vehicle_df.head()


## Step 5. Review Saved Artifact

This final cell previews the annotated frame written to disk.

In [ ]:
saved_frame = cv2.imread(str(ANNOTATED_FRAME_PATH))
saved_frame_rgb = cv2.cvtColor(saved_frame, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 5))
plt.imshow(saved_frame_rgb)
plt.title('Annotated Detection Output')
plt.axis('off')
plt.show()
